Day 17: Advanced Filtering and Conditional Selection

This notebook covers Compound Boolean Masking, Bitwise Precedence, `.isin()` Membership, Vectorized String Filtering, and Expression Evaluation with `.query()`

## Validation goals
- Master element-wise boolean masking using bitwise operators (`&`, `|`, `~`) and enforce explicit parenthetical precedence[cite: 12]
- Apply efficient collection membership filtering using `.isin()`[cite: 12]
- Perform vectorized text pattern matching using the `.str` accessor (`.str.contains()`, `.str.startswith()`)[cite: 12]
- Write expressive, SQL-like query strings using `.query()` and interpolate external Python variables with `@`[cite: 12]
- Analyze execution order errors and memory/performance considerations of vectorized operations vs. custom Python loops[cite: 12]

Filtering and selecting subsets of data based on dynamic conditions is one of the most frequent operations in tabular data analysis. In Pandas, filtering relies on Boolean Masking, where conditions evaluate to arrays of `True` and `False` values that determine which rows or columns are retained.

Moving beyond basic bracket filtering `(df[df['col'] > 5])`, production-grade pipelines leverage bitwise operators, string accessor methods, membership tests, and expression parsing via `.query()` to write clean, maintainable, and highly optimized code.

### Core Concepts & Deep Dive

1. Boolean Masking & Bitwise Operators
    - When applying multiple conditions in Pandas, standard Python logical operators (`and`, `or`, `not`) fail because they evaluate truth values for the entire object rather than element-wise.

    - Bitwise Operators: Use `&` (`AND`), `|` (`OR`), and `~` (`NOT`) for element-wise array evaluation.

    - Operator Precedence: Python's bitwise operators have higher precedence than comparison operators (e.g., `>`). Therefore, parentheses around each individual condition are mandatory (e.g., (`df['amount'] > 100) & (df['status'] == 'active'`)). Without parentheses, Python evaluates the bitwise expression incorrectly, raising a ValueError or TypeError.

2. The `.query()` Method for Readability
    For complex filtering rules involving multiple variables, the `.query()` method parses string expressions directly:

    - Syntax: `df.query("amount > 100 and store_id == @target_store")`
    Local Variable Interpolation: Using the `@` symbol allows referencing local Python variables inside the query string cleanly without messy string concatenation.

3. Membership & String Accessors (`.isin()` and `.str`)
    - Membership Filtering `(.isin()):` Checking if column values belong to a collection (list, set, or Series) is efficiently handled using `.isin()`, which acts as a vectorized SQL IN clause.

    - String Filtering (`.str` accessor): Pandas exposes string manipulation methods through the `.str` namespace. Methods like `.str.contains()`, `.str.startswith()`, `.str.lower()`, and regex matching allow vectorized text filtering without slow Python loops.

### Hands-On Implementation, Filtering & Selection:

Here is a comprehensive script demonstrating various filtering techniques on a simulated transactional dataset:

In [ ]:
import numpy as np
import pandas as pd

# Setup mock dataset
np.random.seed(42)
n_rows = 1000

df = pd.DataFrame({
    "txn_id": [f"TXN_{i:04d}" for i in range(n_rows)],
    "category": np.random.choice(["Electronics", "Apparel", "Groceries", "Hardware"], size=n_rows),
    "amount": np.round(np.random.uniform(10.0, 1500.0, size=n_rows), 2),
    "store_region": np.random.choice(["North-Region", "South-Region", "East-Region", "West-Region"], size=n_rows),
    "is_flagged": np.random.choice([True, False], size=n_rows, p=[0.1, 0.9])
})

# ==========================================
# 1. Boolean Masking with Bitwise Operators
# ==========================================
# Requirement: High-value electronics in North or East regions
# Note the mandatory parentheses around each condition!
mask_complex = (df["amount"] > 500.0) & (df["category"] == "Electronics") & (df["store_region"].str.contains("North|East"))
filtered_mask_df = df[mask_complex]

print(f"--- Boolean Mask Filter (Rows: {len(filtered_mask_df)}) ---")
print(filtered_mask_df.head(3))


# ==========================================
# 2. Membership Filtering using .isin()
# ==========================================
# Requirement: Transactions belonging to specific target categories
target_categories = ["Groceries", "Apparel"]
filtered_isin_df = df[df["category"].isin(target_categories)]

print(f"\n--- .isin() Filter (Rows: {len(filtered_isin_df)}) ---")
print(filtered_isin_df["category"].value_counts())


# ==========================================
# 3. String Methods & Vectorized Text Filtering
# ==========================================
# Requirement: Store regions starting with 'South' and unflagged transactions
filtered_str_df = df[df["store_region"].str.startswith("South") & (~df["is_flagged"])]

print(f"\n--- String & Negation Filter (Rows: {len(filtered_str_df)}) ---")
print(filtered_str_df[["txn_id", "store_region", "is_flagged"]].head(3))


# ==========================================
# 4. Expression Parsing via .query()
# ==========================================
# Requirement: Amount greater than 1000 and matching an external variable threshold
threshold = 1200.0
filtered_query_df = df.query("amount > @threshold and category == 'Hardware'")

print(f"\n--- .query() Method Filter (Rows: {len(filtered_query_df)}) ---")
print(filtered_query_df)

### Guided Question

***Operator Precedence & Syntax Errors***

If you write a compound filter expression as `df[df['amount'] > 100 & df['category'] == 'Electronics']` without parentheses around the individual comparisons, Pandas raises a ValueError.

Why does Python evaluate bitwise operators (&) before comparison operators (>), and how do explicit parentheses resolve this execution order?

The issue occurs because of Python's operator precedence: bitwise & has higher precedence than comparison operators such as > and ==.

1. Why the expression fails
Your expression is:

`df[df['amount'] > 100 & df['category'] == 'Electronics']`

Python evaluates `&` before the comparisons, so it does not interpret the expression as two independent conditions joined together. Instead, the expression is grouped in a way that can trigger an invalid operation or an ambiguous truth-value error when Pandas processes the resulting Series.

**The correct expression is:**

`df[
    (df['amount'] > 100) &
    (df['category'] == 'Electronics')
]`

The parentheses explicitly group each comparison before applying `&`.
- `(df['amount'] > 100)` produces a Boolean Series indicating which amounts exceed `100`.
- `(df['category'] == 'Electronics')` produces a Boolean Series indicating which rows belong to the Electronics category.
- `&` combines the two Boolean Series element by element, returning `True` only where both conditions are satisfied.

3. Why use `&` instead of `and`?
Pandas Series contain multiple Boolean values. The `&` operator performs `element-wise` logical `AND`, while Python's `and` expects single truth values and cannot directly combine two Boolean Series.

Rule to remember: Always put parentheses around each comparison when combining Pandas conditions with `&` or `|`.

### Guided Question

***Memory & Vectorized Text Matching***

Suppose you need to filter a dataframe for rows where a product description contains a specific keyword case-insensitively.

Why is using `df['description'].str.lower().str.contains('laptop')` computationally preferred over writing a custom Python for loop with an `if` statement `checking str.lower()` on every row?

Using `df['description'].str.lower().str.contains('laptop')` is generally preferred because it uses `Pandas`' vectorized string operations, avoiding the overhead of manually iterating over rows in Python.
Why it is more efficient
1. Less Python-level overhead: A custom for loop repeatedly executes Python instructions for every row. Pandas applies its string operations across the Series through optimized internal implementations where available.
2. Concise and readable: One expression processes the entire column and produces a Boolean Series, rather than requiring you to build and maintain a separate list of results.
3. Easy filtering: The Boolean Series can be used directly to select matching rows.

In [ ]:
# Vectorized approach
mask = df['description'].str.lower().str.contains('laptop', na=False)
filtered_df = df[mask]

Here, `na=False` treats missing descriptions as non-matches instead of allowing missing values to propagate into the mask.

***Important nuance***
Pandas string operations are not guaranteed to be faster in every situation. Depending on the string dtype and implementation, some operations still involve Python-level processing internally. However, the vectorized approach is generally cleaner and often more efficient than a manually written Python loop.

Key takeaway: Prefer Pandas vectorized operations for column-wide transformations and filtering, while remembering that vectorization does not automatically guarantee native-code execution or the fastest possible performance.

### Guided Question:

**Choosing Between Bracket Filtering and `.query()`**

In what scenarios would you choose the `.query()` method over traditional boolean bracket indexing `(df[...])`, and what are the readability advantages of using the `@` symbol inside query strings?

You would choose `Pandas`' `.query()` method when filtering a DataFrame using multiple conditions, especially when the expression becomes long or involves several columns.
1. When to use `.query()` vs. boolean indexing
Traditional boolean indexing is useful for straightforward filters and when you need to build or reuse Boolean masks.

In [ ]:
filtered_df = df[
    (df["amount"] > 100) &
    (df["category"] == "Electronics")
]

Using `.query()` can make compound conditions easier to read because you can write column names directly without repeatedly typing `df["..."]` or adding parentheses around each comparison.

In [ ]:
filtered_df = df.query(
    "amount > 100 and category == 'Electronics'"
)

Both expressions select rows where the amount exceeds 100 and the category is Electronics.

Choose `.query()` when:
- You have several filtering conditions.
- You want a concise, SQL-like expression.
- Your column names are valid Python identifiers, making them easy to reference.
Prefer boolean indexing when:
- You already have a Boolean mask.
- Your conditions require complex Python functions or custom logic.
- You want explicit, step-by-step control over the filtering process.
2. What does the `@` symbol do inside `.query()`?
The `@` symbol lets you reference a Python variable defined outside the query string.

For example:

In [ ]:
minimum_amount = 100
target_category = "Electronics"

filtered_df = df.query(
    "amount > @minimum_amount and category == @target_category"
)

Here:
- `@minimum_amount` refers to the external Python variable holding 100.
- `@target_category` refers to the external variable holding "Electronics".
Without `@,` Pandas generally interprets those names as DataFrame column names or other names in the query's evaluation environment, rather than as the external variables you intended.
Why this improves readability
It separates your filtering logic from your filter parameters. You can change the threshold or category without rewriting the query expression.

It also avoids manually constructing query strings with string concatenation or f-strings, which can introduce quoting problems and potential expression-injection risks when values come from untrusted input.

Key takeaway: Use `.query()` for readable, SQL-like compound filters, and use `@variable` to insert existing Python variables into the expression.